In [3]:
import xarray as xr
import numpy as np
import pandas as pd
import netCDF4 as nc
import os
from datetime import datetime, timedelta
import glob
import tqdm

In [4]:
# Base directories
gruan_base_dir = '/home/chinahg/GCresearch/GRUAN_sondes/'
era5_base_dir = '/home/chinahg/GCresearch/ERA5_downloads/'

# Function to construct ERA5 file path from GRUAN file path
def construct_era5_path(gruan_file_path):
    date_str = os.path.basename(gruan_file_path).split('_')[4][:8]
    era5_file_path = os.path.join(era5_base_dir, date_str[:4], f'{date_str[:4]}_{date_str[4:6]}_{date_str[6:8]}.nc')
    return era5_file_path

# Arrays to store the paths
gruan_file_paths = []
era5_file_paths = []

# Recursively find all GRUAN files and construct corresponding ERA5 file paths
for root, dirs, files in os.walk(gruan_base_dir):
    for file in files:
        if file.endswith('.nc'):
            gruan_file_path = os.path.join(root, file)
            era5_file_path = construct_era5_path(gruan_file_path)
            gruan_file_paths.append(gruan_file_path)
            era5_file_paths.append(era5_file_path)

# Sort the paths
gruan_file_paths.sort()
era5_file_paths.sort()

In [5]:
####################################################################################################
### Functions ###

def fill_nan_with_next(arr):
    """Fill NaN values in the array with the next non-NaN value, if available."""
    for i in range(len(arr) - 1):
        if np.isnan(arr[i]):
            next_valid = next((x for x in arr[i + 1:] if not np.isnan(x)), np.nan)
            arr[i] = next_valid
    return arr

def press2alt(pressure):
    """
    Convert pressure to altitude.

    Parameters
    ----------
    pressure : Union[int, np.ndarray]
        Pressure in Pascal.

    Returns
    -------
    Union[float, np.ndarray]
        Altitude in meters.
    """
    L = -6.5*10**-3
    P0 = 101325
    T0 = 288.15
    R = 287.053
    g = 9.81

    altitudes = np.zeros_like(pressure)

    if type(pressure)==int:
        return (T0/L)*((pressure*100/P0)**(-R*L/g) -1)
    else:
        for i in range(len(pressure)):
            altitudes[i] = (T0/L)*((pressure[i]*100/P0)**(-R*L/g) -1)

        return altitudes

def alt2press(altitude):
    """
    Convert altitude to pressure.

    Parameters
    ----------
    altitude : Union[float, np.ndarray]
        Altitude in meters.

    Returns
    -------
    Union[int, np.ndarray]
        Pressure in Pascal.
    """
    L = -6.5*10**-3
    P0 = 101325
    T0 = 288.15
    R = 287.053
    g = 9.81

    if isinstance(altitude, (float, np.float32)):
        return P0*(1 + L*altitude/T0)**(-g/(R*L))
    else:
        pressures = np.zeros_like(altitude)
        for i in range(len(altitude)):
            pressures[i] = P0*(1 + L*altitude[i]/T0)**(-g/(R*L))

        return pressures

def calculate_evaporation_depth(altitude_diff):
    # Placeholder for actual evaporation depth logic
    return altitude_diff * 0.5  # Example: just a simple dummy calculation

def calculate_MLD(altitudes, humidities):
    # Convert arrays to read as floats
    humidities = np.array(humidities, dtype=float)
    altitudes = np.array(altitudes, dtype=float)
    print("Humidities shape: ", humidities.shape)
    print("Altitudes shape: ", altitudes.shape)
    
    # Initialize the binary array to store the regime classification
    regime_array = [0] * len(altitudes)  # Start by assuming all are subsaturated (0)

    # Step 1: Find the first altitude with RH >= 1 (supersaturated)
    found_supersaturated = False
    first_supersaturated_index = -1
    
    for i in range(len(humidities)):
        if humidities[i] >= 1:
            found_supersaturated = True
            first_supersaturated_index = i
            regime_array[i] = 1  # Mark this altitude as supersaturated
            break  # Stop once we find the first supersaturated altitude

    if not found_supersaturated:
        print("No supersaturated altitudes found.")
    else:
        print(f"First supersaturated altitude found at {altitudes[first_supersaturated_index]}m.")

        # Step 2: Now classify the rest of the altitudes
        for i in range(first_supersaturated_index + 1, len(humidities)):
            if humidities[i] >= 1:
                regime_array[i] = 1  # Mark as supersaturated (since RH >= 1)

        # Step 3: For altitudes below the first supersaturated, calculate evaporation depth
        for i in range(first_supersaturated_index - 1, -1, -1):
            if humidities[i] < 1:
                # Calculate evaporation depth between current altitude and first supersaturated altitude
                altitude_diff = altitudes[first_supersaturated_index] - altitudes[i]
                evaporation_depth = calculate_evaporation_depth(altitude_diff)

                # If the evaporation depth is sufficient, mark as supersaturated
                if evaporation_depth >= altitude_diff:
                    regime_array[i] = 1
                else:
                    regime_array[i] = 0  # It's subsaturated
    return regime_array

In [16]:
# Define the dimensions
days = pd.date_range('2005-01-01', '2021-12-31')  # From 2005 to the end of 2021
E_latitudes = np.linspace(30, 60, int((60 - 30) / 0.25) + 1)  # 0.25 degree increments between 30 and 60 degrees
E_longitudes = np.linspace(-180, 180, int(360 / 0.25) + 1)  # 0.25 degree increments
iterator = np.linspace(0, 1, 1, dtype=int)  # PLACEHOLDER, FOR DEBUGGING
print(iterator)

# Initialize lists to store data
valid_combinations = []
G_data_list = []
E_RHi = []

for j in tqdm.tqdm(iterator):  # Iterate through the dates
    # Open the ERA5 file
    E_file_path = era5_file_paths[j]
    E_data = xr.open_dataset(E_file_path)

    # Open the GRUAN file
    G_file_path = gruan_file_paths[j]
    G_data = nc.Dataset(G_file_path)

    # Extract the base time from the G_data attributes
    base_time_str = G_data.variables['time'].units.split('since ')[1]
    base_time = datetime.strptime(base_time_str, '%Y-%m-%dT%H:%M:%S')

    G_datetime = [base_time + timedelta(seconds=float(sec)) for sec in G_data.variables['time'][:]]  # Convert the time variable from seconds since base_time to datetime objects
    G_site = os.path.basename(G_file_path).split('_')[0].split('-')[0]
    G_lat = fill_nan_with_next(G_data.variables['lat'][:])
    print("G_lat: ", G_lat)
    G_lon = fill_nan_with_next(G_data.variables['lon'][:])
    G_alt = fill_nan_with_next(G_data.variables['alt'][:])
    G_RHi = G_data.variables['rh_i'][:]
    G_MLD = calculate_MLD(G_alt, G_RHi)

    E_datetime = E_data.variables['time'][:].values
    E_alt = np.array(press2alt(E_data.sel(latitude=G_lat[0], longitude=G_lon[0], time=G_datetime[0], method='nearest')['isobaricInhPa']))

    # Have to average over the GRUAN data to regrid it to ERA5 size 
    indexer = len(G_alt)
    for i in range(indexer):
        E_RHi.append(np.array(E_data.sel(latitude=G_lat[i], longitude=G_lon[i], isobaricInhPa=alt2press(G_alt[i]), time=G_datetime[i], method='nearest')['RH_i']))

    # Find the index ranges for where the G_alt values fall within the E_alt values
    index_ranges = []
    for i in range(len(G_alt)):
        for j in range(len(E_alt) - 1):
            if G_alt[i] >= E_alt[j] and G_alt[i] < E_alt[j + 1]:
                index_ranges.append(j)
                break
        if G_alt[i] >= E_alt[-1]:
            index_ranges.append(j)

    print("Index ranges for regridding:", index_ranges)

    # Average the RHi values in E_RHi based on index_ranges
    E_RHi_avg = []
    for i in range(len(E_alt)):
        indices = [idx for idx, val in enumerate(index_ranges) if val == i]
        if indices:
            avg_rhi = np.mean([E_RHi[idx] for idx in indices])
            E_RHi_avg.append(avg_rhi)
        else:
            E_RHi_avg.append(np.nan)  # If no indices found, append NaN

    E_RHi_avg = np.array(E_RHi_avg)
    print("E_RHi_avg shape: ", E_RHi_avg.shape)

    E_MLD = calculate_MLD(E_alt, E_RHi_avg)

    # Create a dictionary for the current data
    current_data = {
        'G_site': G_site,
        'G_lat': G_lat,
        'G_lon': G_lon,
        'G_alt': G_alt,
        'G_RHi': G_RHi,
        'G_MLD': G_MLD,
        'G_dt': G_datetime,
        'E_alt': E_alt,
        'E_RHi': E_RHi_avg,
        'E_MLD': E_MLD,
        'E_dt': E_datetime
    }

    G_data_list.append(current_data)

    # Create a list of non-empty (day, lat, lon) combinations
    E_latitude = float(E_data.latitude.sel(latitude=G_lat[0], method='nearest').values)
    E_longitude = float(E_data.longitude.sel(longitude=G_lon[0], method='nearest').values)

    combo = (G_datetime[0].strftime('%Y-%m-%d'), E_latitude, E_longitude)
    valid_combinations.append(combo)
    print("Valid combinations appended!")

# Convert G_data_list to a DataFrame
df = pd.DataFrame(G_data_list)

# Convert valid combinations into a MultiIndex
index = pd.MultiIndex.from_tuples(valid_combinations, names=['day', 'E_latitude', 'E_longitude'])

# Check for duplicates in the MultiIndex
duplicates = index.duplicated(keep=False)

# Report duplicates if any
if duplicates.any():
    print("Duplicates found in index:")
    print(index[duplicates])
else:
    print("No duplicates found in index.")

# Set the MultiIndex to the DataFrame
df.set_index(index, inplace=True)

# Save the DataFrame to a parquet file
df.to_parquet('final_data.parquet')


[0]


  0%|          | 0/1 [00:00<?, ?it/s]

G_lat:  [39.053837 39.053837 39.053837 ... 39.139282 39.13922  39.139175]
Humidities shape:  (5925,)
Altitudes shape:  (5925,)
No supersaturated altitudes found.


100%|██████████| 1/1 [00:16<00:00, 16.60s/it]

Index ranges for regridding: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6

In [ ]:
%load_ext line_profiler

In [ ]:
%load_ext line_profiler

In [ ]:
%lprun -f calculate_MLD calculate_MLD(E_alt, E_RHi)

In [17]:
df

,,,G_site,G_lat,G_lon,G_alt,G_RHi,G_MLD,G_dt,E_alt,E_RHi,E_MLD,E_dt
day,E_latitude,E_longitude,,,,,,,,,,,
2012-10-16,39.0,-77.0,BEL,"[39.053837, 39.053837, 39.053837, 39.053837, 3...","[-76.87736, -76.87736, -76.87736, -76.87736, -...","[54.546295, 58.966248, 62.898277, 66.26977, 70...","[0.8860545, 0.8557449, 0.83857447, 0.81740654,...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[2012-10-16 07:03:42, 2012-10-16 07:03:43.0001...","[110.84666010462223, 323.27149394987333, 540.1...","[33.774742126464844, 40.54938507080078, 40.549...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[2005-06-08T00:00:00.000000000, 2005-06-08T01:..."


In [ ]:
import matplotlib.pyplot as plt

# Define the target altitude
target_altitude = 12000  # 12 km

# Function to find the nearest index
def find_nearest_index(array, value):
    idx = (np.abs(array - value)).argmin()
    return idx

# Extract the GRUAN and ERA5 relative humidity values at the nearest altitude to 12km
gruan_rh_12km = []
era5_rh_12km = []

for i in range(len(df)):
    print(i)
    # Find the nearest altitude index for GRUAN
    gruan_alt_index = find_nearest_index(np.ma.getdata(df['G_alt'].iloc[i]), target_altitude)
    
    # Find the nearest time index for ERA5
    gruan_time = df['G_dt'].iloc[i][gruan_alt_index]
    era5_time_index = find_nearest_index(df['E_dt'].iloc[i], gruan_time)
    
    # Find the nearest altitude index for ERA5 at the matched time
    era5_alt_index = find_nearest_index(df['E_alt'].iloc[i], target_altitude)
    
    # Append the relative humidity values at the nearest altitude and time
    gruan_rh_12km.append(df['G_RHi'].iloc[i][gruan_alt_index])
    era5_rh_12km.append(df['E_RHi'].iloc[i].values[era5_time_index, era5_alt_index])

print(np.shape(gruan_rh_12km))
print(np.shape(era5_rh_12km))

# Create the scatter plot
plt.figure(figsize=(10, 6))
plt.scatter(gruan_rh_12km, era5_rh_12km, alpha=0.5, edgecolors='k')
plt.xlabel('GRUAN Relative Humidity at 12km')
plt.ylabel('ERA5 Relative Humidity at 12km')
plt.title('Scatter Plot of GRUAN vs ERA5 Relative Humidity at 12km')
plt.grid(True)
plt.show()

In [ ]:
dataset

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Define the 4 points (x1, y1), (x2, y2), (x3, y3), (x4, y4)
x1, y1, z1 = 1, 1, 10  # Point 1
x2, y2, z2 = 2, 1, 20  # Point 2
x3, y3, z3 = 1, 2, 30  # Point 3
x4, y4, z4 = 2, 2, 40  # Point 4

# Define the query point (x, y) inside the rectangle formed by the 4 points
x_query = 1.1
y_query = 1.9

# Step 1: Interpolate along the x-direction (between points 1 and 2, 3 and 4)
def lerp(x0, x1, y0, y1, x):
    return y0 + (x - x0) * (y1 - y0) / (x1 - x0)

# Interpolate in the x-direction for y = 1 (between points (x1, y1) and (x2, y2))
z_left = lerp(x1, x2, z1, z2, x_query)

# Interpolate in the x-direction for y = 2 (between points (x3, y3) and (x4, y4))
z_right = lerp(x3, x4, z3, z4, x_query)

# Step 2: Interpolate in the y-direction (between the results from the previous step)
z_result = lerp(y1, y3, z_left, z_right, y_query)

# Plotting the points and the interpolation process

fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(111)

# Plot the corner points as red markers
ax.scatter([x1, x2, x3, x4], [y1, y2, y3, y4], color='red', label='Corners')

# Annotate the corner points
ax.text(x1, y1, f'({x1}, {y1}) = {z1}', fontsize=12, verticalalignment='bottom', horizontalalignment='right')
ax.text(x2, y2, f'({x2}, {y2}) = {z2}', fontsize=12, verticalalignment='bottom', horizontalalignment='left')
ax.text(x3, y3, f'({x3}, {y3}) = {z3}', fontsize=12, verticalalignment='top', horizontalalignment='right')
ax.text(x4, y4, f'({x4}, {y4}) = {z4}', fontsize=12, verticalalignment='top', horizontalalignment='left')

# Plot the query point
ax.scatter(x_query, y_query, color='blue', label=f'Query Point ({x_query}, {y_query})')

# Draw lines connecting the query point to the interpolation steps
ax.plot([x1, x_query], [y1, y_query], 'k--', linewidth=1)
ax.plot([x2, x_query], [y2, y_query], 'k--', linewidth=1)
ax.plot([x3, x_query], [y3, y_query], 'k--', linewidth=1)
ax.plot([x4, x_query], [y4, y_query], 'k--', linewidth=1)

# Add a colorbar for the interpolation grid (optional)
grid_x = np.linspace(1, 2, 100)
grid_y = np.linspace(1, 2, 100)
grid_X, grid_Y = np.meshgrid(grid_x, grid_y)

# Create a grid of interpolated values
Z = np.array([[lerp(x1, x2, lerp(y1, y3, z1, z3, y), lerp(y2, y4, z2, z4, y), x) 
              for x in grid_x] for y in grid_y])

# Plot the interpolation as a heatmap
c = ax.pcolormesh(grid_X, grid_Y, Z, shading='auto', cmap='coolwarm', alpha=0.5)
fig.colorbar(c, ax=ax, label='Interpolated Value')

# Labels and title
ax.set_xlabel('X')
ax.set_ylabel('Y')
ax.set_title('2D Linear Interpolation')

# Show the plot with the query point and the interpolation heatmap
ax.legend()
plt.show()

# Output interpolated result
print(f"The interpolated value at ({x_query}, {y_query}) is {z_result}")


In [ ]:
import matplotlib.pyplot as plt

# Plotting the altitude vs humidity
plt.figure(figsize=(10, 6))

# Plot humidity
plt.plot(humidities, altitudes, label='Relative Humidity', marker='o')

# Highlight supersaturated points
supersaturated_altitudes = [altitudes[i] for i in range(len(regime_array)) if regime_array[i] == 1]
supersaturated_humidities = [humidities[i] for i in range(len(regime_array)) if regime_array[i] == 1]
plt.scatter(supersaturated_humidities, supersaturated_altitudes, color='red', label='Supersaturated', zorder=5)

# Add a red dashed line where relative humidity = 1
plt.axvline(x=1, color='red', linestyle='--', label='RH = 1')

# Add labels and title
plt.xlabel('Relative Humidity')
plt.ylabel('Altitude (m)')
plt.title('Altitude vs Relative Humidity with Regime Classification')
plt.legend()
plt.grid(True)

# Show the plot
plt.show()